# mini-llm-runner: Building Generative LLM from Scratch
**Author:** Vikash Mehta ([@vkshdev](https://github.com/vkshdev))  
**Project:** build-mini-LLM
---

### Pipeline Structure
1. **Phase 1 (Pre-training):** Foundational language modeling on `roneneldan/TinyStories` (~160M tokens) for vocabulary, syntax, and narrative coherence.
2. **Phase 2 (Conversational SFT):** Supervised instruction tuning on `tatsu-lab/alpaca` with ChatML formatting and causal next-token loss masking.
3. **Inference & Benchmarking:** Quantitative throughput testing (tokens/second) and qualitative generation inspection across both base and aligned checkpoints.

## 1. Checking Hardware & Runtime Environment

In [ ]:
import os
import sys
import torch

print(f"Runtime: Python {sys.version.split()[0]} | PyTorch {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_count = torch.cuda.device_count()
    print(f"Detected {gpu_count} GPU device(s):")
    for i in range(gpu_count):
        props = torch.cuda.get_device_properties(i)
        print(f"  Device {i}: {props.name} | VRAM: {props.total_memory / (1024**3):.2f} GB | Compute: {props.major}.{props.minor}")
else:
    print("[Alert] No GPU detected. Enable GPU accelerator in Kaggle notebook settings.")

!nvidia-smi

## 2. Cloning Codebase & Setting Up Dependencies

In [ ]:
import os

REPO_URL = "https://github.com/vkshdev/build-mini-LLM.git"
PROJECT_DIR = "/kaggle/working/build-mini-LLM"
CHECKPOINT_DIR = "/kaggle/working/checkpoints"

if not os.path.exists(PROJECT_DIR):
    print(f"Cloning repository from {REPO_URL}...")
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    print("Repository exists. Pulling latest commits...")
    !git -C {PROJECT_DIR} pull origin main

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

%cd {PROJECT_DIR}
!pip install -q -r requirements.txt
print("[OK] Environment prepared.")

## 3. Authenticating Weights & Biases Dashboard

In [ ]:
import wandb

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    my_wandb_key = secrets.get_secret("WANDB_API_KEY")
    wandb.login(key=my_wandb_key)
    print("[OK] Authenticated with W&B via Kaggle Secrets.")
except Exception:
    print("Kaggle secret not detected. Falling back to W&B login:")
    wandb.login()

## 4. Phase 1: Pre-training Base 28M LLM on TinyStories
Pre-train the 28.3M parameter decoder-only Transformer across ~160M tokens using FP16 mixed precision.

In [ ]:
!python -m src.train \
    --data_dir /kaggle/working/build-mini-LLM/data \
    --checkpoint_dir /kaggle/working/checkpoints \
    --max_steps 10000 \
    --batch_size 32 \
    --lr 6e-4 \
    --resume True

## 5. Phase 2: Conversational Fine-Tuning (Chat SFT)
Align the pre-trained weights into an interactive assistant using ChatML formatting and causal next-token loss masking.

In [ ]:
!python -m src.sft_train \
    --base_model /kaggle/working/checkpoints/best_pretrained_28m.pt \
    --output_dir /kaggle/working/checkpoints \
    --max_steps 1500 \
    --batch_size 16 \
    --lr 3e-5

## 6. Testing & Benchmarking Model Outputs
Evaluate generation quality and measure throughput (tokens/second) across both pre-trained story continuations and conversational instruction responses.

In [ ]:
import os
import time
import torch
from src.config import ModelConfig
from src.tokenizer import get_tokenizer
from src.model import SmallLLM

device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = get_tokenizer()
stop_ids = [tokenizer.im_end_id, tokenizer.eos_id]

# Part A: Evaluate Base Pre-trained Model (Story Generation)
pretrained_path = "/kaggle/working/checkpoints/best_pretrained_28m.pt"
if os.path.exists(pretrained_path):
    print(f"Evaluating Base Pre-trained Model: {pretrained_path}")
    base_model = SmallLLM(ModelConfig()).to(device)
    ckpt = torch.load(pretrained_path, map_location=device)
    base_model.load_state_dict(ckpt.get("model_state_dict", ckpt))
    base_model.eval()

    story_prompts = [
        "Once upon a time in a sunny meadow,",
        "A little girl named Lily found a shiny silver key and"
    ]
    for sp in story_prompts:
        tokens = torch.tensor(tokenizer.encode(sp), dtype=torch.long, device=device).unsqueeze(0)
        t0 = time.time()
        out = base_model.generate(tokens, max_new_tokens=60, temperature=0.75, top_k=35, stop_id=stop_ids)
        dt = time.time() - t0
        new_tokens = out[0, tokens.size(1):].tolist()
        story = tokenizer.decode(new_tokens).replace("<|endoftext|>", "").strip()
        print(f"Prompt:      {sp}")
        print(f"Story:       {sp} {story}")
        print(f"Throughput:  {len(new_tokens)} tokens in {dt:.2f}s ({len(new_tokens)/max(dt,1e-4):.1f} tok/s)\n")

# Part B: Evaluate Conversational Chatbot (Instruction Following)
chat_path = "/kaggle/working/checkpoints/chat_model_28m.pt"
if os.path.exists(chat_path):
    print(f"Evaluating Conversational Chatbot: {chat_path}")
    chat_model = SmallLLM(ModelConfig()).to(device)
    ckpt = torch.load(chat_path, map_location=device)
    chat_model.load_state_dict(ckpt.get("model_state_dict", ckpt))
    chat_model.eval()

    def ask_bot(user_msg: str, max_new_tokens: int = 70):
        prompt_str = f"<|im_start|>user\n{user_msg}<|im_end|>\n<|im_start|>assistant\n"
        tokens = torch.tensor(tokenizer.encode(prompt_str), dtype=torch.long, device=device).unsqueeze(0)
        t0 = time.time()
        out = chat_model.generate(tokens, max_new_tokens=max_new_tokens, temperature=0.7, top_k=30, top_p=0.9, stop_id=stop_ids)
        dt = time.time() - t0
        new_tokens = out[0, tokens.size(1):].tolist()
        reply = tokenizer.extract_reply(tokenizer.decode(new_tokens))
        print(f"User:        {user_msg}")
        print(f"Assistant:   {reply}")
        print(f"Throughput:  {len(new_tokens)} tokens in {dt:.2f}s ({len(new_tokens)/max(dt,1e-4):.1f} tok/s)\n")

    chat_queries = [
        "Can you explain why the sky looks blue in simple words?",
        "Write a 2-sentence story about a friendly mouse sharing an apple.",
        "What should you do if you see a lost dog in the park?"
    ]
    for q in chat_queries:
        ask_bot(q)

## 7. Reviewing Checkpoint Artifacts
Inspect the checkpoint file sizes in `/kaggle/working/checkpoints`.

In [ ]:
import glob

my_ckpts = glob.glob("/kaggle/working/checkpoints/*.pt")
if my_ckpts:
    print("Checkpoint artifacts:")
    for p in sorted(my_ckpts):
        size_mb = os.path.getsize(p) / (1024 * 1024)
        print(f"  {os.path.basename(p):<28} {size_mb:8.2f} MB")
else:
    print("No checkpoint files found.")